# Model Latency Test

Test latency characteristics for a model hosted in Azure AI Foundry.

**Metrics captured:** Time to First Token (TTFT), Total Response Time, Tokens/sec

## 1. Setup

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from utils import get_client, get_model_name, get_openai_client, run_latency_test, run_responses_latency_test

## 2. Configure Model Endpoint

Reads from `.env` by default. Use `responses` for Azure OpenAI / Foundry OpenAI base URLs such as `*.openai.azure.com` or endpoints ending in `/openai/v1`. Use `chat_completions` for the Chat Completions API on the same client.

In [ ]:
# Use "responses" for the Responses API (recommended for OpenAI models).
# Use "chat_completions" for the Chat Completions API.
API_TYPE = "responses"

# Override the endpoint inline if needed:
# endpoint = "https://your-resource.openai.azure.com"
endpoint = None

# --- Experiment knobs ---
NUM_ITERATIONS = 3       # Repeat each prompt N times for statistical power
WARMUP = 1               # Throwaway requests before measured runs
REASONING_EFFORT = None  # "low" | "medium" | "high" | None (for reasoning models)
MAX_OUTPUT_TOKENS = 256  # Cap output length for controlled comparison (None = unlimited)

if API_TYPE == "responses":
    client = get_openai_client(endpoint=endpoint)
    run_test = run_responses_latency_test
elif API_TYPE == "chat_completions":
    client = get_client(endpoint=endpoint)
    run_test = run_latency_test
else:
    raise ValueError("API_TYPE must be 'responses' or 'chat_completions'")

model = get_model_name()

print(f"API type:          {API_TYPE}")
print(f"Model:             {model}")
print(f"Iterations:        {NUM_ITERATIONS}")
print(f"Warmup requests:   {WARMUP}")
print(f"Reasoning effort:  {REASONING_EFFORT}")
print(f"Max output tokens: {MAX_OUTPUT_TOKENS}")

API type: inference
Model: gpt-4o-mini


## 3. Define Test Prompts

Add, remove, or modify prompts to vary the test. A mix of short and long expected outputs is useful.

In [3]:
prompts = [
    "What is the capital of France?",
    "Explain quantum computing in 3 sentences.",
    "Write a Python function that checks if a string is a palindrome.",
    "Summarize the key ideas of machine learning in a short paragraph.",
]

## 4. Run Latency Measurements

In [ ]:
results = run_test(
    client, model, prompts,
    num_iterations=NUM_ITERATIONS,
    warmup=WARMUP,
    reasoning_effort=REASONING_EFFORT,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)
print(f"Completed {len(results)} measurements ({len(prompts)} prompts × {NUM_ITERATIONS} iterations)")

Completed 4 measurements


## 5. Results

In [ ]:
df = pd.DataFrame(results)
display_cols = [
    "iteration", "prompt_index", "prompt", "ttft", "total_time",
    "tokens_per_second", "decode_tokens_per_second", "completion_tokens",
]
if df["reasoning_tokens"].any():
    display_cols.append("reasoning_tokens")
if df["reasoning_effort"].notna().any():
    display_cols.append("reasoning_effort")
df[display_cols]

In [ ]:
import numpy as np

print("=== Summary ===")
print(f"Iterations:             {NUM_ITERATIONS}")
print(f"Avg TTFT:               {df['ttft'].mean():.4f}s")
print(f"Median TTFT:            {df['ttft'].median():.4f}s")
print(f"P95 TTFT:               {df['ttft'].quantile(0.95):.4f}s")
print(f"Std TTFT:               {df['ttft'].std():.4f}s")
print(f"Avg Total Time:         {df['total_time'].mean():.4f}s")
print(f"Median Total Time:      {df['total_time'].median():.4f}s")
print(f"P95 Total Time:         {df['total_time'].quantile(0.95):.4f}s")
print(f"Std Total Time:         {df['total_time'].std():.4f}s")
print(f"Avg Tokens/sec (e2e):   {df['tokens_per_second'].mean():.2f}")
print(f"Avg Tokens/sec (decode):{df['decode_tokens_per_second'].mean():.2f}")
print(f"Total Tokens:           {df['completion_tokens'].sum()}")
if df["reasoning_tokens"].any():
    print(f"Total Reasoning Tokens: {df['reasoning_tokens'].sum()}")

## 6. Visualize (Optional)

In [ ]:
import matplotlib.pyplot as plt

metrics = ["ttft", "total_time", "tokens_per_second", "decode_tokens_per_second"]
titles = ["Time to First Token (s)", "Total Response Time (s)", "Tokens/sec (e2e)", "Tokens/sec (decode)"]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, metric, title in zip(axes, metrics, titles):
    if NUM_ITERATIONS > 1:
        df.boxplot(column=metric, by="prompt_index", ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Prompt")
        plt.suptitle("")  # remove default boxplot title
    else:
        ax.bar(df["prompt_index"], df[metric])
        ax.set_title(title)
        ax.set_xlabel("Prompt")

plt.tight_layout()
plt.show()